# 02 — FastGAN @ 256px (main model)```bashpython -m src.train --name fastgan --steps 100000python -m src.train --name fastgan --resume ../checkpoints/fastgan/latest.pt```~37 h on a 4 GB RTX 3050 at batch 8, peaking at 2.72 GiB. Checkpoints every2,000 steps — resume is first-class because the run has to survive beingstopped nightly.

In [ ]:
import sys; sys.path.insert(0, "..")from pathlib import Pathimport torchimport matplotlib.pyplot as pltfrom tensorboard.backend.event_processing.event_accumulator import EventAccumulator

In [ ]:
def load_events(run):    ea = EventAccumulator(str(sorted(Path(f"../runs/{run}").glob("events.*"))[-1]))    ea.Reload()    return eadef plot_tags(ea, tags):    fig, axes = plt.subplots(1, len(tags), figsize=(4.7 * len(tags), 3.4))    for tag, ax in zip(tags, [axes] if len(tags) == 1 else axes):        pts = ea.Scalars(tag)        ax.plot([p.step for p in pts], [p.value for p in pts])        ax.set_title(tag); ax.set_xlabel("step")        ax.spines[["top", "right"]].set_visible(False)    plt.tight_layout(); plt.show()def plot_accuracy(ea):    """Both curves pinned at ~1.0 means D has memorised the training set."""    fig, ax = plt.subplots(figsize=(7, 3.2))    for tag in ("train/acc_real", "train/acc_fake"):        pts = ea.Scalars(tag)        ax.plot([p.step for p in pts], [p.value for p in pts], label=tag)    ax.axhline(1.0, ls="--", c="crimson", lw=1, label="memorisation")    ax.set_ylim(0, 1.05); ax.set_xlabel("step"); ax.legend()    ax.spines[["top", "right"]].set_visible(False)    plt.tight_layout(); plt.show()

## Architecture, shapes and parameter counts

In [ ]:
from src.config import FastGANConfigfrom src.models.fastgan import build_modelscfg = FastGANConfig()G, D = build_models(cfg.z_dim, cfg.ngf, cfg.ndf, cfg.resolution)print(f"G params: {sum(p.numel() for p in G.parameters()) / 1e6:.1f}M")print(f"D params: {sum(p.numel() for p in D.parameters()) / 1e6:.1f}M")big, small = G(torch.randn(2, cfg.z_dim))print("G outputs:", tuple(big.shape), tuple(small.shape))logits, recons, part = D(torch.randn(2, 3, 256, 256), real=True)print("D logits:", tuple(logits.shape))print("reconstructions:", [tuple(r.shape) for r in recons], "| quadrant:", part)

## Skip-Layer ExcitationThe gate is a 4x4-pooled conv producing one sigmoid weight per channel of thehigh-res map. Long-range conditioning — 8x8 structure modulating 128x128detail — for almost no parameters, which is what keeps this inside 4 GB.

In [ ]:
gate = G.se_128.main(torch.randn(1, G.se_128.main[1].in_channels, 8, 8))print("SLE gate shape:", tuple(gate.shape), "-> one multiplier per channel")print(f"se_128 params: {sum(p.numel() for p in G.se_128.parameters()) / 1e3:.1f}K")print(f"feat_128 params: {sum(p.numel() for p in G.feat_128.parameters()) / 1e6:.2f}M")

## What DiffAugment actually doesApplied to reals **and** fakes in **both** the G and D passes. The ops aredifferentiable, so gradients reach G and it never learns to reproduce theartefacts. Applying it only to reals is the classic bug — it just hands D afree tell.

In [ ]:
from src.augment import diff_augmentfrom src.data.dataset import PokemonArtworkds = PokemonArtwork(resolution=256, mirror=False)batch = torch.stack([ds[i] for i in range(4)])aug = diff_augment(batch, ("color", "translation", "cutout"))fig, axes = plt.subplots(2, 4, figsize=(13, 6.8))for i in range(4):    axes[0, i].imshow(((batch[i].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)); axes[0, i].axis("off")    axes[1, i].imshow(((aug[i].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)); axes[1, i].axis("off")axes[0, 0].set_title("real", loc="left"); axes[1, 0].set_title("DiffAugment", loc="left")plt.tight_layout(); plt.show()

## Training curves

In [ ]:
ea = load_events("fastgan")plot_tags(ea, ["train/d", "train/g", "train/recon"])

### The overfitting tellIf both accuracy curves sit at ~1.0 for a sustained stretch, D has memorisedthe set. On this dataset that means DiffAugment is misapplied or theself-supervised reconstruction term is not doing its job — stop and fix itrather than training longer.

In [ ]:
plot_accuracy(ea)

## Progress animation

In [ ]:
from src.sample import training_giftraining_gif(Path("../samples/fastgan"), Path("../docs/assets/training.gif"))